In [1]:
import csv
from pathlib import Path


DATA_PATH = Path("data") / "purchases.csv"
OUTPUT_PATH = Path("output") / "recommendations.csv"

In [14]:
def read_purchases(path: Path = DATA_PATH) -> tuple[list[dict], list[dict]]:
    """Return (valid_rows, rejected_rows). Decide what makes a purchase row unusable."""
    valid_rows = []
    rejected_rows = []

    with open(path, newline="", encoding="utf-8") as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            # Check for missing or empty fields
            if not row.get("customer_id") or not row.get("product_id"):
                rejected_rows.append(row)
                continue

            # Check for invalid data (e.g., negative quantities)
            if not row.get("quantity") or int(row["quantity"]) < 0:
                rejected_rows.append(row)
                continue

            # If all checks pass, add to valid rows
            valid_rows.append(row)
    return valid_rows, rejected_rows

In [ ]:

def jaccard(a: set, b: set) -> float:
    """Return Jaccard similarity using set operators."""
    union = a | b
    if not union:
        return 1.0
    return len(a & b) / len(union)


In [ ]:
class RecommendationEngine:
    """
    Recommends products a customer has not bought, based on similar customers.
    In RecommendationEngine.__init__, build a dict of customer → set of product IDs, plus lookups 
    for product names/categories and product popularity. Build them once; later 
    methods must not re-scan the raw rows
    """

    def __init__(self, purchases: list[dict], min_similarity: float = 0.0) -> None:
        # Build derived indexes once; later methods need not scan raw rows.
        self.min_similarity = min_similarity
        self.customer_products: dict[str, set[str]] = {}
        self.product_info: dict[str, dict[str, str]] = {}
        self.product_popularity: dict[str, int] = {}

        for purchase in purchases:
            customer_id = purchase["customer_id"]
            product_id = purchase["product_id"]

            self.customer_products.setdefault(customer_id, set()).add(product_id)
            self.product_popularity[product_id] = (
                self.product_popularity.get(product_id, 0) + 1
            )
            self.product_info.setdefault(
                product_id,
                {
                    "name": purchase.get("product_name", ""),
                    "category": purchase.get("category", ""),
                },
            )


    def history(self, customer_id: str) -> set[str]:
        """Return the set of product IDs bought by this customer, or an empty set if unknown."""
        return self.customer_products.get(customer_id, set())



    def similarity(self, customer_a: str, customer_b: str) -> float:
        """Return a similarity score between two customers, or 0.0 if either is unknown."""

        products_a = self.customer_products.get(customer_a)
        products_b = self.customer_products.get(customer_b)

        if products_a is None or products_b is None:
            return 0.0

        return jaccard(products_a, products_b)


    def similar_customers(self, customer_id: str, k: int = 3) -> list[tuple[str, float]]:
        """Return the top-k similar other customers, best first."""
        if customer_id not in self.customer_products or k <= 0:
            return []

        similarities = []
        for other_id in self.customer_products:
            if other_id == customer_id:
                continue

            score = self.similarity(customer_id, other_id)
            if score >= self.min_similarity:
                similarities.append((other_id, score))

        similarities.sort(key=lambda item: (-item[1], item[0]))
        return similarities[:k]


In [30]:

# Read the purchase data
valid_purchases, rejected_purchases = read_purchases()

engine = RecommendationEngine(valid_purchases)

# engine.similarity("C110", "C119")


engine.similar_customers("C110", k=3)


[('C114', 0.5), ('C102', 0.25), ('C103', 0.25)]

In [ ]:
def similar_customers(self, customer_id: str, k: int = 3) -> list[tuple[str, float]]:
    """Return the k most similar other customers as (customer_id, score), best first."""
    raise NotImplementedError

def candidate_products(self, customer_id: str, k: int = 3) -> dict[str, dict]:
    """Products bought by similar customers but not by this one, with the evidence for each."""
    raise NotImplementedError

def score(self, evidence: dict) -> float:
    """Your ranking rule. Combine similarity with at least one other signal."""
    raise NotImplementedError

def recommend(self, customer_id: str, n: int = 3) -> list[dict]:
    """Return up to n ranked recommendations for one customer."""
    raise NotImplementedError

def recommend_all(self, n: int = 3) -> list[dict]:
    raise NotImplementedError

def export_csv(self, rows: list[dict], path: Path = OUTPUT_PATH) -> None:
    raise NotImplementedError
